# Osoba A: BERT (HerBERT) + PEFT

Zakres (Część I, ~3,25 pkt): I.A, pkt 1, 2 i 4 z [instrukcji](../Zadanie_2_zmodyfikowane.md).

## Do zrobienia

1. **I.A**: zapoznać się z klasami `BertModel`, `BertForMaskedLM`, `BertForSequenceClassification`, `BertForTokenClassification` (krótka notatka w notebooku: co zwracają, jaka głowica, jakie wejścia).
2. **Pkt 1: strojenie z adapterem PEFT** (np. LoRA), model `allegro/herbert-base-cased`:
   - a. klasyfikacja tekstu (`BertForSequenceClassification`),
   - b. klasyfikacja tokenów (`BertForTokenClassification`),
   - c. ocena na zbiorze testowym.
3. **Pkt 2: wizualizacja przestrzeni wektorowej** zbioru testowego (tylko klasyfikacja tekstu), **przed** i **po** dostrojeniu. Reprezentacja tekstu = wektor tokenu `[CLS]`. Dwa interaktywne wykresy kolorowane etykietą i opis w markdownie, jak strojenie zmienia układ punktów.
4. **Pkt 4: augmentacja przez `[MASK]`** (`BertForMaskedLM`): maskujemy losowe słowa w tekstach treningowych i podstawiamy predykcje modelu. Potem trenujemy model na zbiorze powiększonym.

## Ustalone (nie zmieniamy bez uzgodnienia z B)

- Model: `allegro/herbert-base-cased`.
- Etykiety tekstu: `plus_m, minus_m, zero, amb` (`LABELS`, `LABEL2ID`). Tagi tokenów: BIO `wzmocnienie / oslabienie / negacja` (`TAGS`, `TAG2ID`).
- **Zbiór testowy jest wspólny z osobą B i nigdy nie jest augmentowany.**
  - tekst: `load_text('test')` (`all.text.test.txt`); zbiór treningowy: `load_text('train')`, walidacyjny: `load_text('dev')`,
  - tokeny: `split_tokens(load_tokens())` (seed 42).
- Augmentacja dotyczy tylko zbioru treningowego. Wynik zapisujemy w `../data/aug_bert.jsonl` (pola `text`, `label`, gdzie `label` to nazwa etykiety, a nie liczba). Finalny model trenujemy na `train + aug`.
- Adaptery i checkpointy zapisujemy w `../artifacts/` (nie w repo).

## Części wspólne (`../utils/`)

| Co | Skąd |
|---|---|
| wczytanie tekstów, etykiety | `utils.data`: `load_text`, `LABELS`, `LABEL2ID` |
| tagi BIO i podział train/test dla tokenów | `utils.tokens`: `load_tokens`, `split_tokens`, `TAGS`, `TAG2ID` |
| tagi słów → subtokeny (`-100`) | `utils.align`: `encode_tokens` |
| wykres przestrzeni wektorowej | `utils.plots`: `plot_embeddings` |

## Uwagi

- **Klasyfikacja tokenów ma mało danych**: anotacje z Zadania 1 to tylko 35 tekstów, więc wyniki będą niestabilne. Opisz to w wnioskach. Rozszerzenie (np. słownik z anotacji) ustalamy osobno z zespołem.
- Metryki: tekst → accuracy i macro-F1 (`sklearn`); tokeny → precision/recall/F1 per tag, z pominięciem pozycji `-100`.
- Wizualizacja: wektor `[CLS]` z ostatniej warstwy ukrytej (przed głowicą klasyfikacyjną). Dla modelu z adapterem PEFT weź go z modelu po `merge`/z wyłączonymi głowicami, tak jak dla bazowego.


In [2]:
import sys

sys.path.insert(0, "..")
from utils.align import encode_tokens
from utils.data import LABEL2ID, LABELS, load_text
from utils.plots import plot_embeddings
from utils.tokens import TAG2ID, TAGS, load_tokens, split_tokens

MODEL = "allegro/herbert-base-cased"
train, dev, test = load_text("train"), load_text("dev"), load_text("test")
tok_train, tok_test = split_tokens(load_tokens())
len(train), len(dev), len(test), len(tok_train), len(tok_test)

(6573, 823, 820, 28, 7)

## I.A: klasy BERT (notatka)

Wszystkie cztery klasy dzielą to samo ciało `model.bert` (`BertModel`). Różnią się tylko małą głowicą na wierzchu i funkcją straty. Wymiary podane są dla `herbert-base-cased`: 12 warstw, `hidden=768`, słownik 50 000 tokenów. `B` to batch, `T` to liczba subtokenów (z `[CLS]`, czyli `<s>`, i `[SEP]`, czyli `</s>`, oraz paddingiem).

**Wejścia (wspólne):** `input_ids` `(B, T)`, `attention_mask` `(B, T)` (1 = prawdziwy token, 0 = padding) i opcjonalnie `token_type_ids`. Klasy z głowicą przyjmują też `labels`. Jeśli je podamy, zwracają `loss`.

| Klasa | Głowica | `logits` / wyjście | `labels` → `loss` |
|---|---|---|---|
| `BertModel` | brak (tylko enkoder) + **pooler**: `tanh(Linear(768→768))` na wektorze `[CLS]` | `last_hidden_state` `(B, T, 768)`, `pooler_output` `(B, 768)` | brak |
| `BertForMaskedLM` | `BertOnlyMLMHead`: `Linear(768→768)` → GELU → LayerNorm → `decoder Linear(768→50000)` (wagi związane z embeddingami wejściowymi). Bez poolera. | `(B, T, 50000)`: rozkład po słowniku **dla każdej pozycji** | `(B, T)`, id prawdziwego tokenu na pozycjach `[MASK]`, `-100` w pozostałych. CrossEntropy po słowniku. |
| `BertForSequenceClassification` | `pooler_output` → dropout → `Linear(768→num_labels)` | `(B, 4)`: jeden wektor na **cały tekst** (u nas `plus_m, minus_m, zero, amb`) | `(B,)` z id klas. CrossEntropy (MSE, gdy `num_labels=1`; BCE dla multi-label). |
| `BertForTokenClassification` | `last_hidden_state` → dropout → `Linear(768→num_labels)`. Bez poolera. | `(B, T, 7)`: wektor klas **dla każdego subtokenu** (u nas 7 tagów BIO) | `(B, T)`, `-100` na `[CLS]`, `[SEP]`, paddingu i nie-pierwszych subtokenach słowa (`utils.align.encode_tokens`). CrossEntropy z `ignore_index=-100`. |

**Uwagi praktyczne:**
- `logits` to surowe wyniki, a nie prawdopodobieństwa. Klasa to `argmax`, a prawdopodobieństwa daje `softmax`.
- Przy `num_labels`, który różni się od checkpointu, `classifier` jest losowo inicjalizowany (ostrzeżenie „newly initialized”). Trzeba go trenować, a przy PEFT oznacza to dodanie go do `modules_to_save`.
- Do pkt 2 bierzemy surowy `[CLS]` = `last_hidden_state[:, 0]`, **a nie** `pooler_output` (ten jest dodatkowo przepuszczony przez Linear i tanh).
- W tokenizerze HerBERT `[CLS]`, `[SEP]` i `[MASK]` mają postać `<s>`, `</s>` i `<mask>`. W kodzie używamy `tok.cls_token`, `tok.mask_token` itd.

In [5]:
import torch
from transformers import (
    AutoTokenizer,
    BertForMaskedLM,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertModel,
)

tok = AutoTokenizer.from_pretrained(MODEL)
batch = tok(
    ["Hotel był bardzo czysty.", "Nie polecam."], padding=True, return_tensors="pt"
)
B, T = batch["input_ids"].shape

with torch.no_grad():
    out = BertModel.from_pretrained(MODEL)(**batch)
    assert out.last_hidden_state.shape == (B, T, 768)
    assert out.pooler_output.shape == (B, 768)
    for cls, kw, shape in [
        (BertForMaskedLM, {}, (B, T, tok.vocab_size)),
        (BertForSequenceClassification, {"num_labels": len(LABELS)}, (B, len(LABELS))),
        (BertForTokenClassification, {"num_labels": len(TAGS)}, (B, T, len(TAGS))),
    ]:
        m = cls.from_pretrained(MODEL, **kw)
        assert m(**batch).logits.shape == shape
        print(
            f"{cls.__name__:32} logits {tuple(shape)}  głowica: {getattr(m, 'cls', None) or m.classifier}"
        )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.sso.sso_relationship.weight            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.sso.sso_relationship.bias              | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: allegro/herbert-base-cased
Key                             | Status     |  | 
--------------------------------+------------+--+-
cls.sso.sso_relationship.weight | UNEXPECTED |  | 
cls.sso.sso_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias          | UNEXPECTED |  | 
bert.pooler.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM                  logits (2, 7, 50000)  głowica: BertOnlyMLMHead(
  (predictions): BertLMPredictionHead(
    (transform): BertPredictionHeadTransform(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (transform_act_fn): GELUActivation()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    )
    (decoder): Linear(in_features=768, out_features=50000, bias=True)
  )
)


...

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

BertForSequenceClassification    logits (2, 4)  głowica: Linear(in_features=768, out_features=4, bias=True)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be

BertForTokenClassification       logits (2, 7, 7)  głowica: Linear(in_features=768, out_features=7, bias=True)


## 1a. Klasyfikacja tekstu + PEFT

HerBERT + LoRA (`query`, `value`, `alpha = 2r`) dla trzech rang `r ∈ {4, 8, 16}`. Wspólny trening: `utils/train.py` (`max_length=400`, 3 epoki, `lr=2e-4`). Najlepszy checkpoint każdego przebiegu i najlepsze `r` wybieramy po **macro-F1 na dev**, a test służy tylko do końcowej oceny (pkt 1c). Adaptery zapisujemy w `../artifacts/bert-lora-r{r}/best`.

In [ ]:
from pathlib import Path

import pandas as pd
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification

from utils.train import make_dataset, plot_history
from utils.train import train as fit


ART = Path("../artifacts")


def make_model(r):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL,
        num_labels=len(LABELS),
        id2label=dict(enumerate(LABELS)),
        label2id=LABEL2ID,
    )
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=2 * r,
        lora_dropout=0.1,
        target_modules=["query", "value"],
    )
    return get_peft_model(
        model, cfg
    )  # SEQ_CLS: `classifier` trafia do modules_to_save, więc też się trenuje


MAX_LEN = 400
tok = AutoTokenizer.from_pretrained(MODEL)
train_ds, dev_ds, test_ds = (make_dataset(d, tok, MAX_LEN) for d in (train, dev, test))

Map:   0%|          | 0/6573 [00:00<?, ? examples/s]

Map:   0%|          | 0/823 [00:00<?, ? examples/s]

Map:   0%|          | 0/820 [00:00<?, ? examples/s]

In [ ]:
RANKS = [4, 8, 16]
EPOCHS = 3
results, histories, preds = [], {}, {}
for r in RANKS:
    model = make_model(r)
    model.print_trainable_parameters()
    trainer = fit(model, tok, train_ds, dev_ds, ART / f"bert-lora-r{r}", epochs=EPOCHS)
    trainer.save_model(ART / f"bert-lora-r{r}" / "best")
    out = trainer.predict(test_ds)
    histories[f"r={r}"] = trainer.state.log_history
    preds[r] = out.predictions.argmax(-1)
    results.append(
        {
            "r": r,
            "trainable_params": sum(
                p.numel() for p in model.parameters() if p.requires_grad
            ),
            "train_time_s": next(
                log["train_runtime"]
                for log in trainer.state.log_history
                if "train_runtime" in log
            ),
            "dev_f1": trainer.state.best_metric,
            "test_accuracy": out.metrics["test_accuracy"],
            "test_f1": out.metrics["test_f1"],
        }
    )

results = pd.DataFrame(results)
results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

Epoch,Training Loss,Validation Loss


In [ ]:
plot_history(histories, "HerBERT + LoRA: krzywe uczenia")

### 1aa. Ocena na zbiorze testowym (najlepsze `r` wg dev)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best = int(results.loc[results["dev_f1"].idxmax(), "r"])
print(f"najlepsze r (wg dev macro-F1) = {best}\n")
print(
    classification_report(test["label_id"], preds[best], target_names=LABELS, digits=3)
)
ConfusionMatrixDisplay.from_predictions(
    test["label_id"], preds[best], display_labels=LABELS, cmap="Blues"
);

**Wnioski 1a/1c** _(do uzupełnienia po pełnym treningu)_: wpływ `r` na jakość i czas, przebieg krzywych (czy jest przeuczenie), które klasy są mylone (`amb`?).

## 1b. Klasyfikacja tokenów + PEFT

## 2. Wizualizacja przestrzeni wektorowej (`[CLS]`): przed i po strojeniu

## 4. Augmentacja przez `[MASK]` i trening na zbiorze powiększonym